# Projet 11 — Préparation des données pour l’étude des marchés d’exportation

Ce notebook construit une table avec une ligne par pays pour préparer l’étude des marchés d’exportation de poulet. Les données alimentaires et économiques portent sur **2017** ; la population de **2000** sert à calculer la croissance démographique sur la période 2000–2017.

La catégorie alimentaire retenue est **« Viande de Volailles »** : elle couvre un périmètre plus large que le seul poulet.

Le traitement suit cinq étapes : préparation des données alimentaires, préparation de la population, rapprochement des données FAO, ajout des indicateurs de la Banque mondiale, puis contrôle et export.

## <span style="color: blue;">1. Préparer les données de disponibilité alimentaire</span>


In [2]:
import pandas as pd
import numpy as np


### <span style="color: DeepSkyBlue;">1.1. Importer les données et examiner leur structure</span>

Les fichiers CSV sources doivent se trouver dans le répertoire de travail du notebook. Les contrôles ci-dessous portent sur les dimensions, les types de colonnes et les valeurs distinctes.

In [3]:
dispo = pd.read_csv("DisponibiliteAlimentaire_2017.csv", sep=",", encoding="utf-8-sig")

In [4]:
dispo.shape

(176600, 14)

In [5]:
dispo.head(3)

,Code Domaine,Domaine,Code zone,Zone,Code Élément,Élément,Code Produit,Produit,Code année,Année,Unité,Valeur,Symbole,Description du Symbole
0,FBS,Nouveaux Bilans Alimentaire,2,Afghanistan,5511,Production,2511,Blé et produits,2017,2017,Milliers de tonnes,4281.0,S,Données standardisées
1,FBS,Nouveaux Bilans Alimentaire,2,Afghanistan,5611,Importations - Quantité,2511,Blé et produits,2017,2017,Milliers de tonnes,2302.0,S,Données standardisées
2,FBS,Nouveaux Bilans Alimentaire,2,Afghanistan,5072,Variation de stock,2511,Blé et produits,2017,2017,Milliers de tonnes,-119.0,S,Données standardisées


In [6]:
dispo.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 176600 entries, 0 to 176599
Data columns (total 14 columns):
 #   Column                  Non-Null Count   Dtype  
---  ------                  --------------   -----  
 0   Code Domaine            176600 non-null  object 
 1   Domaine                 176600 non-null  object 
 2   Code zone               176600 non-null  int64  
 3   Zone                    176600 non-null  object 
 4   Code Élément            176600 non-null  int64  
 5   Élément                 176600 non-null  object 
 6   Code Produit            176600 non-null  int64  
 7   Produit                 176600 non-null  object 
 8   Code année              176600 non-null  int64  
 9   Année                   176600 non-null  int64  
 10  Unité                   176600 non-null  object 
 11  Valeur                  176600 non-null  float64
 12  Symbole                 176600 non-null  object 
 13  Description du Symbole  176600 non-null  object 
dtypes: float64(1), int64

In [7]:
dispo.nunique()

Code Domaine                 1
Domaine                      1
Code zone                  174
Zone                       174
Code Élément                17
Élément                     17
Code Produit                98
Produit                     98
Code année                   1
Année                        1
Unité                        4
Valeur                    7250
Symbole                      2
Description du Symbole       2
dtype: int64

La table de disponibilité alimentaire contient **176 600 lignes et 14 colonnes**, pour **174 zones** et **98 produits**.

### <span style="color: DodgerBlue;">1.2. Sélectionner les colonnes utiles et la viande de volaille</span>

Aucune valeur manquante n’est présente dans les 14 colonnes de la table source. Les colonnes « Code Domaine », « Domaine », « Code année » et « Année » sont constantes dans ce fichier. Les colonnes « Symbole » et « Description du Symbole » décrivent le statut des données ; elles ne sont pas utilisées dans cette analyse.

La suppression de ces six colonnes laisse **8 colonnes**. L’année de référence, **2017**, reste précisée dans ce notebook.

In [8]:
# Retirer les colonnes constantes et les métadonnées non utilisées dans cette analyse.
colonnes_a_supprimer = [ "Code Domaine", "Domaine", "Code année",
    "Année", "Symbole", "Description du Symbole"]
dispo_clean = dispo.drop(columns=colonnes_a_supprimer)

In [9]:
dispo_clean.shape

(176600, 8)

La suppression de ces six colonnes laisse **8 colonnes**. L’année de référence, **2017** est précisée pour l'ensemble de co notebook (sauf population de 2000).

Parmi les **98 produits**, nous retenons « Viande de Volailles », la catégorie disponible la plus proche du produit étudié. Ce filtre conserve **2 061 lignes** et **8 colonnes**.

In [10]:
volaille = dispo_clean[dispo_clean["Produit"] == "Viande de Volailles"]

In [11]:
volaille.shape

(2061, 8)

In [12]:
volaille.head()

,Code zone,Zone,Code Élément,Élément,Code Produit,Produit,Unité,Valeur
651,2,Afghanistan,5511,Production,2734,Viande de Volailles,Milliers de tonnes,28.0
652,2,Afghanistan,5611,Importations - Quantité,2734,Viande de Volailles,Milliers de tonnes,29.0
653,2,Afghanistan,5072,Variation de stock,2734,Viande de Volailles,Milliers de tonnes,0.0
654,2,Afghanistan,5301,Disponibilité intérieure,2734,Viande de Volailles,Milliers de tonnes,57.0
655,2,Afghanistan,5123,Pertes,2734,Viande de Volailles,Milliers de tonnes,2.0


In [13]:
volaille.info()

<class 'pandas.core.frame.DataFrame'>
Index: 2061 entries, 651 to 176400
Data columns (total 8 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Code zone     2061 non-null   int64  
 1   Zone          2061 non-null   object 
 2   Code Élément  2061 non-null   int64  
 3   Élément       2061 non-null   object 
 4   Code Produit  2061 non-null   int64  
 5   Produit       2061 non-null   object 
 6   Unité         2061 non-null   object 
 7   Valeur        2061 non-null   float64
dtypes: float64(1), int64(3), object(4)
memory usage: 144.9+ KB


In [14]:
volaille.groupby("Élément")["Unité"].unique()

Élément
Alimentation pour touristes                                      [Milliers de tonnes]
Aliments pour animaux                                            [Milliers de tonnes]
Autres utilisations (non alimentaire)                            [Milliers de tonnes]
Disponibilité alimentaire (Kcal/personne/jour)                   [Kcal/personne/jour]
Disponibilité alimentaire en quantité (kg/personne/an)                           [kg]
Disponibilité de matière grasse en quantité (g/personne/jour)       [g/personne/jour]
Disponibilité de protéines en quantité (g/personne/jour)            [g/personne/jour]
Disponibilité intérieure                                         [Milliers de tonnes]
Exportations - Quantité                                          [Milliers de tonnes]
Importations - Quantité                                          [Milliers de tonnes]
Nourriture                                                       [Milliers de tonnes]
Pertes                                        

### <span style="color: blueviolet;">1.3. Choisir les indicateurs alimentaires

La catégorie « Viande de Volailles » couvre **172 pays et territoires**. Le nombre de zones renseignées varie selon l’élément : une table source sans cellule vide peut néanmoins omettre certaines combinaisons de zone et d’élément. Le contrôle suivant mesure cette couverture.

In [15]:
volaille.groupby("Élément")["Zone"].nunique().sort_values(ascending=False)

Élément
Disponibilité alimentaire en quantité (kg/personne/an)           172
Disponibilité de protéines en quantité (g/personne/jour)         172
Disponibilité alimentaire (Kcal/personne/jour)                   172
Disponibilité de matière grasse en quantité (g/personne/jour)    172
Disponibilité intérieure                                         170
Nourriture                                                       170
Importations - Quantité                                          170
Variation de stock                                               169
Production                                                       168
Résidus                                                          164
Exportations - Quantité                                          135
Alimentation pour touristes                                       78
Pertes                                                            67
Traitement                                                        46
Autres utilisations (non a

Nous retenons **8 indicateurs** décrivant la production, les importations, la disponibilité intérieure et la disponibilité alimentaire. Ils sont renseignés pour **168 à 172 zones** chacun. Les autres éléments ne sont pas conservés à cette étape, notamment les exportations, disponibles pour 135 zones.

In [16]:
elements_a_garder = [
    "Production",
    "Importations - Quantité",
    "Disponibilité intérieure",
    "Nourriture",
    "Disponibilité alimentaire en quantité (kg/personne/an)",
    "Disponibilité alimentaire (Kcal/personne/jour)",
    "Disponibilité de protéines en quantité (g/personne/jour)",
    "Disponibilité de matière grasse en quantité (g/personne/jour)"
]

In [17]:
# Créer une copie indépendante pour modifier la sélection sans affecter la table source.
volaille_selection = volaille[
    volaille["Élément"].isin(elements_a_garder)
].copy()

In [18]:
volaille_selection.head()

,Code zone,Zone,Code Élément,Élément,Code Produit,Produit,Unité,Valeur
651,2,Afghanistan,5511,Production,2734,Viande de Volailles,Milliers de tonnes,28.00
652,2,Afghanistan,5611,Importations - Quantité,2734,Viande de Volailles,Milliers de tonnes,29.00
654,2,Afghanistan,5301,Disponibilité intérieure,2734,Viande de Volailles,Milliers de tonnes,57.00
657,2,Afghanistan,5142,Nourriture,2734,Viande de Volailles,Milliers de tonnes,55.00
658,2,Afghanistan,645,Disponibilité alimentaire en quantité (kg/pers...,2734,Viande de Volailles,kg,1.53


In [19]:
volaille_selection.shape

(1366, 8)

### <span style="color: CornflowerBlue;">1.4. Organiser la table avec une ligne par zone</span>

La sélection contient **1 366 lignes** avant le pivot. Les unités sont intégrées aux noms des futurs indicateurs, puis le pivot produit **172 lignes et 10 colonnes** : 2 identifiants et 8 indicateurs alimentaires.

In [20]:
# Inclure les unités dans les noms des huit indicateurs avant le pivot.

noms_variables = {
    "Production": "Production (milliers de tonnes)",
    "Importations - Quantité": "Importations (milliers de tonnes)",
    "Disponibilité intérieure": "Disponibilité intérieure (milliers de tonnes)",
    "Nourriture": "Nourriture (milliers de tonnes)",
    "Disponibilité alimentaire en quantité (kg/personne/an)": "Disponibilité alimentaire (kg/personne/an)",
    "Disponibilité alimentaire (Kcal/personne/jour)": "Disponibilité alimentaire (kcal/personne/jour)",
    "Disponibilité de protéines en quantité (g/personne/jour)": "Protéines (g/personne/jour)",
    "Disponibilité de matière grasse en quantité (g/personne/jour)": "Matières grasses (g/personne/jour)"
}

volaille_selection["Variable"] = volaille_selection["Élément"].map(noms_variables)

In [21]:
volaille_selection.head()

,Code zone,Zone,Code Élément,Élément,Code Produit,Produit,Unité,Valeur,Variable
651,2,Afghanistan,5511,Production,2734,Viande de Volailles,Milliers de tonnes,28.00,Production (milliers de tonnes)
652,2,Afghanistan,5611,Importations - Quantité,2734,Viande de Volailles,Milliers de tonnes,29.00,Importations (milliers de tonnes)
654,2,Afghanistan,5301,Disponibilité intérieure,2734,Viande de Volailles,Milliers de tonnes,57.00,Disponibilité intérieure (milliers de tonnes)
657,2,Afghanistan,5142,Nourriture,2734,Viande de Volailles,Milliers de tonnes,55.00,Nourriture (milliers de tonnes)
658,2,Afghanistan,645,Disponibilité alimentaire en quantité (kg/pers...,2734,Viande de Volailles,kg,1.53,Disponibilité alimentaire (kg/personne/an)


In [22]:
volaille_pays = volaille_selection.pivot(
    index=["Code zone", "Zone"],
    columns="Variable",
    values="Valeur"
).reset_index()

In [23]:
volaille_pays.head(3)

Variable,Code zone,Zone,Disponibilité alimentaire (kcal/personne/jour),Disponibilité alimentaire (kg/personne/an),Disponibilité intérieure (milliers de tonnes),Importations (milliers de tonnes),Matières grasses (g/personne/jour),Nourriture (milliers de tonnes),Production (milliers de tonnes),Protéines (g/personne/jour)
0,1,Arménie,54.0,16.06,47.0,35.0,3.39,47.0,11.0,5.44
1,2,Afghanistan,5.0,1.53,57.0,29.0,0.33,55.0,28.0,0.54
2,3,Albanie,85.0,16.36,47.0,38.0,6.45,47.0,13.0,6.26


In [24]:
volaille_pays.shape

(172, 10)

In [25]:
volaille_pays.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 172 entries, 0 to 171
Data columns (total 10 columns):
 #   Column                                          Non-Null Count  Dtype  
---  ------                                          --------------  -----  
 0   Code zone                                       172 non-null    int64  
 1   Zone                                            172 non-null    object 
 2   Disponibilité alimentaire (kcal/personne/jour)  172 non-null    float64
 3   Disponibilité alimentaire (kg/personne/an)      172 non-null    float64
 4   Disponibilité intérieure (milliers de tonnes)   170 non-null    float64
 5   Importations (milliers de tonnes)               170 non-null    float64
 6   Matières grasses (g/personne/jour)              172 non-null    float64
 7   Nourriture (milliers de tonnes)                 170 non-null    float64
 8   Production (milliers de tonnes)                 168 non-null    float64
 9   Protéines (g/personne/jour)                

Après le pivot, la production présente **4 valeurs manquantes** ; les importations, la disponibilité intérieure et la nourriture en présentent **2 chacune**. Les quatre autres indicateurs sont complets.

Le contrôle suivant vérifie l’unicité du code de zone : aucun doublon n’est attendu pour une table comportant une ligne par zone.

In [26]:
volaille_pays.duplicated(subset=["Code zone"]).sum()

np.int64(0)

## <span style="color:MediumVioletRed;">2. Préparer les données de population</span>

### <span style="color: HotPink;">2.1. Examiner la couverture et sélectionner les années</span>

In [27]:
population = pd.read_csv("Population_2000_2018.csv", sep=",", encoding="utf-8-sig")

In [28]:
population.head(3)

,Code Domaine,Domaine,Code zone,Zone,Code Élément,Élément,Code Produit,Produit,Code année,Année,Unité,Valeur,Symbole,Description du Symbole,Note
0,OA,Séries temporelles annuelles,2,Afghanistan,511,Population totale,3010,Population-Estimations,2000,2000,1000 personnes,20779.953,X,Sources internationales sûres,NaN
1,OA,Séries temporelles annuelles,2,Afghanistan,511,Population totale,3010,Population-Estimations,2001,2001,1000 personnes,21606.988,X,Sources internationales sûres,NaN
2,OA,Séries temporelles annuelles,2,Afghanistan,511,Population totale,3010,Population-Estimations,2002,2002,1000 personnes,22600.770,X,Sources internationales sûres,NaN


In [29]:
population.shape

(4411, 15)

In [30]:
population.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4411 entries, 0 to 4410
Data columns (total 15 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Code Domaine            4411 non-null   object 
 1   Domaine                 4411 non-null   object 
 2   Code zone               4411 non-null   int64  
 3   Zone                    4411 non-null   object 
 4   Code Élément            4411 non-null   int64  
 5   Élément                 4411 non-null   object 
 6   Code Produit            4411 non-null   int64  
 7   Produit                 4411 non-null   object 
 8   Code année              4411 non-null   int64  
 9   Année                   4411 non-null   int64  
 10  Unité                   4411 non-null   object 
 11  Valeur                  4411 non-null   float64
 12  Symbole                 4411 non-null   object 
 13  Description du Symbole  4411 non-null   object 
 14  Note                    258 non-null    

In [31]:
population.nunique()

Code Domaine                 1
Domaine                      1
Code zone                  238
Zone                       238
Code Élément                 1
Élément                      1
Code Produit                 1
Produit                      1
Code année                  19
Année                       19
Unité                        1
Valeur                    4398
Symbole                      2
Description du Symbole       2
Note                         1
dtype: int64

In [32]:
population["Élément"].unique()

array(['Population totale'], dtype=object)

In [33]:
population["Unité"].unique()

array(['1000 personnes'], dtype=object)

In [34]:
population["Année"].unique()

array([2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010,
       2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018])

Le fichier contient uniquement la population totale, exprimée en **milliers d’habitants**, sur les années **2000 à 2018**. Une zone peut donc apparaître sur plusieurs lignes, une par année disponible.

Nous conservons **2017** pour l’alignement avec les données alimentaires et **2000** pour calculer la croissance démographique sur 17 ans. La sélection contient **465 lignes** ; le pivot regroupe **238 zones**, dont **9 sans valeur en 2000** et **2 sans valeur en 2017**. Les contrôles ci-dessous identifient ces zones.

In [35]:
# Conserver 2000 et 2017 pour calculer la croissance démographique sur 17 ans.

population_selection = population[
    population["Année"].isin([2000, 2017])
].copy()

In [36]:
population_pays = population_selection.pivot(
    index=["Code zone", "Zone"],
    columns="Année",
    values="Valeur"
).reset_index()

In [37]:
population_selection.shape

(465, 15)

In [38]:
population_pays.shape

(238, 4)

In [39]:
population_pays.head()

Année,Code zone,Zone,2000,2017
0,1,Arménie,3069.591,2944.791
1,2,Afghanistan,20779.953,36296.113
2,3,Albanie,3129.243,2884.169
3,4,Algérie,31042.235,41389.189
4,5,Samoa américaines,57.821,55.620


In [40]:
population_pays.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 238 entries, 0 to 237
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   Code zone  238 non-null    int64  
 1   Zone       238 non-null    object 
 2   2000       229 non-null    float64
 3   2017       236 non-null    float64
dtypes: float64(2), int64(1), object(1)
memory usage: 7.6+ KB


In [41]:
# Préciser l’année et l’unité des colonnes de population.

population_pays = population_pays.rename(columns={
    2000: "Population 2000 (milliers)",
    2017: "Population 2017 (milliers)"
})

In [42]:
population_pays.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 238 entries, 0 to 237
Data columns (total 4 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   Code zone                   238 non-null    int64  
 1   Zone                        238 non-null    object 
 2   Population 2000 (milliers)  229 non-null    float64
 3   Population 2017 (milliers)  236 non-null    float64
dtypes: float64(2), int64(1), object(1)
memory usage: 7.6+ KB


In [43]:
population_pays.isna().sum()

Année
Code zone                     0
Zone                          0
Population 2000 (milliers)    9
Population 2017 (milliers)    2
dtype: int64

In [44]:
# Afficher les deux zones sans population renseignée en 2017.
population_pays[
    population_pays["Population 2017 (milliers)"].isna()]

Année,Code zone,Zone,Population 2000 (milliers),Population 2017 (milliers)
165,186,Serbie-et-Monténégro,10101.175,NaN
184,206,Soudan (ex),33474.409,NaN


In [45]:
population_pays[
    population_pays["Population 2000 (milliers)"].isna()]

Année,Code zone,Zone,Population 2000 (milliers),Population 2017 (milliers)
228,272,Serbie,NaN,8829.628
229,273,Monténégro,NaN,627.563
230,276,Soudan,NaN,40813.397
231,277,Soudan du Sud,NaN,10910.763
232,278,"Bonaire, Saint-Eustache et Saba",NaN,25.401
233,279,Curaçao,NaN,161.997
234,280,Sint Maarten (partie néerlandaise),NaN,41.444
235,281,Saint-Martin (partie française),NaN,36.560
236,282,Saint-Barthélemy,NaN,9.784


Les libellés des zones sans données incluent notamment la Serbie, le Monténégro, le Soudan et plusieurs territoires. Ils suggèrent des différences de périmètre géographique entre les deux années, sans que le fichier permette d’établir à lui seul la cause de chaque absence.

Les valeurs manquantes sont conservées. Leur incidence sera réévaluée après la jointure avec les données alimentaires.

### <span style="color: DeepPink;">2.2. Calculer la croissance démographique</span>

Le calcul mesure la variation totale entre 2000 et 2017, en pourcentage. Une valeur de population manquante entraîne une croissance manquante.

In [47]:
#63

# Calculez la croissance totale entre 2000 et 2017, plutôt qu'un taux annuel.
population_pays["Croissance population 2000-2017 (%)"] = (
    (population_pays["Population 2017 (milliers)"]
     / population_pays["Population 2000 (milliers)"]) - 1
) * 100

In [49]:
population_pays[
    [
        "Zone",
        "Population 2000 (milliers)",
        "Population 2017 (milliers)",
        "Croissance population 2000-2017 (%)"
    ]
].head(10)

Année,Zone,Population 2000 (milliers),Population 2017 (milliers),Croissance population 2000-2017 (%)
0,Arménie,3069.591,2944.791,-4.065688
1,Afghanistan,20779.953,36296.113,74.668889
2,Albanie,3129.243,2884.169,-7.831734
3,Algérie,31042.235,41389.189,33.331859
4,Samoa américaines,57.821,55.620,-3.806575
5,Andorre,65.390,77.001,17.756538
6,Angola,16395.473,29816.766,81.859749
7,Antigua-et-Barbuda,76.016,95.426,25.534098
8,Argentine,36870.787,43937.140,19.165181
9,Australie,18991.431,24584.620,29.451119


In [48]:
population_pays.head()

Année,Code zone,Zone,Population 2000 (milliers),Population 2017 (milliers),Croissance population 2000-2017 (%)
0,1,Arménie,3069.591,2944.791,-4.065688
1,2,Afghanistan,20779.953,36296.113,74.668889
2,3,Albanie,3129.243,2884.169,-7.831734
3,4,Algérie,31042.235,41389.189,33.331859
4,5,Samoa américaines,57.821,55.620,-3.806575


In [50]:
population_pays.columns.name

'Année'

In [53]:
# La colonne des index a pris le titre « année » lors du pivot. il s’agit de supprimer le titre des colonnes 

population_pays = population_pays.rename_axis(columns=None)

In [52]:
population_pays.head()

,Code zone,Zone,Population 2000 (milliers),Population 2017 (milliers),Croissance population 2000-2017 (%)
0,1,Arménie,3069.591,2944.791,-4.065688
1,2,Afghanistan,20779.953,36296.113,74.668889
2,3,Albanie,3129.243,2884.169,-7.831734
3,4,Algérie,31042.235,41389.189,33.331859
4,5,Samoa américaines,57.821,55.620,-3.806575


## <span style="color: salmon;"> 3. Rapprocher les données FAO et calculer les indicateurs</span>


### <span style="color: OrangeRed;">3.1. Joindre les données alimentaires et démographiques</span>

La jointure utilise le **code de zone** et conserve les **172 zones de la table alimentaire**. Le nom de la zone n’est pas repris depuis la table de population afin d’éviter une colonne redondante. Le paramètre `validate="one_to_one"` vérifie l’unicité de la clé dans chaque table.

In [54]:
population_a_merge = population_pays[
    [
        "Code zone",
        "Population 2000 (milliers)",
        "Population 2017 (milliers)",
        "Croissance population 2000-2017 (%)"
    ]
]

In [55]:
df_pays = volaille_pays.merge(
    population_a_merge,
    on="Code zone",
    how="left",
    validate="one_to_one"
)

In [56]:
df_pays.shape

(172, 13)

In [57]:
df_pays.head()

,Code zone,Zone,Disponibilité alimentaire (kcal/personne/jour),Disponibilité alimentaire (kg/personne/an),Disponibilité intérieure (milliers de tonnes),Importations (milliers de tonnes),Matières grasses (g/personne/jour),Nourriture (milliers de tonnes),Production (milliers de tonnes),Protéines (g/personne/jour),Population 2000 (milliers),Population 2017 (milliers),Croissance population 2000-2017 (%)
0,1,Arménie,54.0,16.06,47.0,35.0,3.39,47.0,11.0,5.44,3069.591,2944.791,-4.065688
1,2,Afghanistan,5.0,1.53,57.0,29.0,0.33,55.0,28.0,0.54,20779.953,36296.113,74.668889
2,3,Albanie,85.0,16.36,47.0,38.0,6.45,47.0,13.0,6.26,3129.243,2884.169,-7.831734
3,4,Algérie,22.0,6.38,277.0,2.0,1.50,264.0,275.0,1.97,31042.235,41389.189,33.331859
4,7,Angola,35.0,10.56,319.0,277.0,2.22,315.0,42.0,3.60,16395.473,29816.766,81.859749


In [58]:
df_pays.isna().sum()

Code zone                                         0
Zone                                              0
Disponibilité alimentaire (kcal/personne/jour)    0
Disponibilité alimentaire (kg/personne/an)        0
Disponibilité intérieure (milliers de tonnes)     2
Importations (milliers de tonnes)                 2
Matières grasses (g/personne/jour)                0
Nourriture (milliers de tonnes)                   2
Production (milliers de tonnes)                   4
Protéines (g/personne/jour)                       0
Population 2000 (milliers)                        3
Population 2017 (milliers)                        0
Croissance population 2000-2017 (%)               3
dtype: int64

La population de 2017 est renseignée pour les **172 zones** conservées. La population de 2000 et la croissance démographique restent manquantes pour **3 zones** : la Serbie, le Monténégro et le Soudan. Le tableau suivant recense les zones présentant au moins une valeur manquante, y compris dans les indicateurs alimentaires.

In [59]:
colonnes_manquantes = df_pays.columns[df_pays.isna().any()].tolist()

df_pays.loc[
    df_pays.isna().any(axis=1),
    ["Zone"] + colonnes_manquantes
]

,Zone,Disponibilité intérieure (milliers de tonnes),Importations (milliers de tonnes),Nourriture (milliers de tonnes),Production (milliers de tonnes),Population 2000 (milliers),Croissance population 2000-2017 (%)
46,Djibouti,3.0,3.0,3.0,NaN,717.584,31.566339
84,République démocratique populaire lao,NaN,NaN,NaN,NaN,5323.700,30.605312
93,Maldives,12.0,12.0,7.0,NaN,279.398,77.668416
160,Ouzbékistan,NaN,NaN,NaN,NaN,24769.955,29.026415
169,Serbie,90.0,12.0,90.0,85.0,NaN,NaN
170,Monténégro,10.0,8.0,10.0,4.0,NaN,NaN
171,Soudan,69.0,2.0,65.0,67.0,NaN,NaN


### <span style="color: tomato;">3.2. Calculer le ratio des importations à la disponibilité intérieure</span>

Le ratio est calculé ainsi :

$$R = \frac{\text{Importations}}{\text{Disponibilité intérieure}} \times 100$$

Les deux grandeurs sont exprimées en milliers de tonnes. La colonne conserve son nom « Part importations dans disponibilité intérieure (%) », mais il s’agit d’un **ratio pouvant dépasser 100 %**, et non d’une part nécessairement comprise entre 0 et 100 %.

In [60]:
# 78
df_pays["Part importations dans disponibilité intérieure (%)"] = (
    df_pays["Importations (milliers de tonnes)"]
    / df_pays["Disponibilité intérieure (milliers de tonnes)"]) * 100

In [61]:
df_pays["Part importations dans disponibilité intérieure (%)"].describe()

count    170.000000
mean      45.803811
std       51.850021
min        0.000000
25%        4.233163
50%       24.749473
75%       80.638298
max      323.928571
Name: Part importations dans disponibilité intérieure (%), dtype: float64

In [62]:
df_pays[["Zone", "Part importations dans disponibilité intérieure (%)"]].sort_values(
    "Part importations dans disponibilité intérieure (%)", ascending=False).head(10)

,Zone,Part importations dans disponibilité intérieure (%)
61,Chine - RAS de Hong-Kong,323.928571
167,Belgique,222.368421
123,Timor-Leste,220.000000
49,Gambie,200.000000
16,Îles Salomon,200.000000
105,Pays-Bas,163.440860
108,Vanuatu,133.333333
89,Chine - RAS de Macao,121.052632
21,Cabo Verde,120.000000
164,Samoa,113.333333


Le ratio atteint environ **324 % à Hong Kong** et **222 % en Belgique**. Un dépassement de 100 % n'est pas une erreur. Un pays peut à la fois produire, importer et exporter beaucoup. Les importations brutes ne représentent donc pas simplement un manque de production nationale. Une partie peut être réexportée ou appartenir à une activité commerciale de hub.

### <span style="color: DarkOrange;">3.3. Calculer les importations par habitant</span>


Les importations sont exprimées en milliers de tonnes et la population en milliers d’habitants. Leur quotient donne des tonnes par habitant ; la multiplication par **1 000** permet d’obtenir des **kg par personne et par an**. Cet indicateur mesure les importations rapportées à la population, et non la consommation individuelle.

In [63]:
# 82
# Calculer les importations par habitant, en kg par personne et par an.
df_pays["Importations par habitant (kg/personne/an)"] = (
    df_pays["Importations (milliers de tonnes)"]
    / df_pays["Population 2017 (milliers)"]) * 1000

In [64]:
# 83
df_pays[["Zone", "Importations (milliers de tonnes)", "Population 2017 (milliers)",
        "Importations par habitant (kg/personne/an)"]].sort_values( 
        "Importations par habitant (kg/personne/an)", ascending=False).head(15)

,Zone,Importations (milliers de tonnes),Population 2017 (milliers),Importations par habitant (kg/personne/an)
61,Chine - RAS de Hong-Kong,907.0,7306.322,124.139068
164,Samoa,17.0,195.352,87.022401
130,Saint-Vincent-et-les Grenadines,9.0,109.827,81.947062
128,Saint-Kitts-et-Nevis,4.0,52.045,76.856566
5,Antigua-et-Barbuda,7.0,95.426,73.355270
55,Grenade,7.0,110.874,63.134730
9,Bahamas,24.0,381.755,62.867546
35,Dominique,4.0,71.458,55.976938
129,Sainte-Lucie,10.0,180.954,55.262663
45,Polynésie française,15.0,276.102,54.327748


In [65]:
df_pays.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 172 entries, 0 to 171
Data columns (total 15 columns):
 #   Column                                               Non-Null Count  Dtype  
---  ------                                               --------------  -----  
 0   Code zone                                            172 non-null    int64  
 1   Zone                                                 172 non-null    object 
 2   Disponibilité alimentaire (kcal/personne/jour)       172 non-null    float64
 3   Disponibilité alimentaire (kg/personne/an)           172 non-null    float64
 4   Disponibilité intérieure (milliers de tonnes)        170 non-null    float64
 5   Importations (milliers de tonnes)                    170 non-null    float64
 6   Matières grasses (g/personne/jour)                   172 non-null    float64
 7   Nourriture (milliers de tonnes)                      170 non-null    float64
 8   Production (milliers de tonnes)                      168 non-null    f

In [66]:
df_pays.head(3)

,Code zone,Zone,Disponibilité alimentaire (kcal/personne/jour),Disponibilité alimentaire (kg/personne/an),Disponibilité intérieure (milliers de tonnes),Importations (milliers de tonnes),Matières grasses (g/personne/jour),Nourriture (milliers de tonnes),Production (milliers de tonnes),Protéines (g/personne/jour),Population 2000 (milliers),Population 2017 (milliers),Croissance population 2000-2017 (%),Part importations dans disponibilité intérieure (%),Importations par habitant (kg/personne/an)
0,1,Arménie,54.0,16.06,47.0,35.0,3.39,47.0,11.0,5.44,3069.591,2944.791,-4.065688,74.468085,11.885394
1,2,Afghanistan,5.0,1.53,57.0,29.0,0.33,55.0,28.0,0.54,20779.953,36296.113,74.668889,50.877193,0.798984
2,3,Albanie,85.0,16.36,47.0,38.0,6.45,47.0,13.0,6.26,3129.243,2884.169,-7.831734,80.851064,13.175372


## <span style="color: MediumSeaGreen;"> 4. Intégrer les indicateurs de la Banque mondiale</span>

### <span style="color: LimeGreen;"> 4.1. Importer et examiner les trois sources </span>

Trois fichiers complètent les données FAO :

- le **PIB par habitant en dollars US courants** ;
- la **population rurale en pourcentage de la population totale**, utilisée pour calculer le taux d’urbanisation ;
- la **stabilité politique**, fournie sous forme de score de gouvernance sur une échelle de **0 à 100** (`GOV_WGI_PV_SC`).

Ce dernier indicateur est un score, pas un pourcentage de population.

In [67]:
pib = pd.read_csv("1-Pib par habitant courant.csv", sep=";", decimal=",", encoding="utf-8-sig")
rural = pd.read_csv("2-rural population.csv", sep=";", decimal=",", encoding="utf-8-sig")
stabilite = pd.read_csv("3-governance.csv", sep=";", decimal=",", encoding="utf-8-sig")

In [68]:
pib.info() 

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 265 entries, 0 to 264
Data columns (total 23 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Country Name    263 non-null    object 
 1   Country Code    265 non-null    object 
 2   Indicator Name  265 non-null    object 
 3   Indicator Code  265 non-null    object 
 4   2000            251 non-null    float64
 5   2001            252 non-null    float64
 6   2002            256 non-null    float64
 7   2003            256 non-null    float64
 8   2004            256 non-null    float64
 9   2005            256 non-null    float64
 10  2006            257 non-null    float64
 11  2007            257 non-null    float64
 12  2008            258 non-null    float64
 13  2009            260 non-null    float64
 14  2010            260 non-null    float64
 15  2011            261 non-null    float64
 16  2012            259 non-null    float64
 17  2013            259 non-null    flo

In [69]:
pib.head(3)

,Country Name,Country Code,Indicator Name,Indicator Code,2000,2001,2002,2003,2004,2005,...,2009,2010,2011,2012,2013,2014,2015,2016,2017,2018
0,Aruba,ABW,PIB par habitant ($ US courants),NY.GDP.PCAP.CD,20681.023030,20740.132580,21307.248250,21949.486000,23700.631990,24171.837130,...,25134.771230,24093.140150,25712.384300,25119.665540,25813.571440,26129.839060,27458.220150,27441.550210,28440.041690,30082.158420
1,NaN,AFE,PIB par habitant ($ US courants),NY.GDP.PCAP.CD,706.728441,625.828860,630.513380,815.433657,989.017115,1126.299204,...,1408.615659,1628.768573,1761.838296,1731.869096,1705.839003,1689.982381,1498.715971,1335.743165,1529.923080,1553.617944
2,Afghanistan,AFG,PIB par habitant ($ US courants),NY.GDP.PCAP.CD,174.930991,138.706822,178.954088,198.871116,221.763654,254.184249,...,452.053705,560.621506,606.694676,651.417134,637.087099,625.054942,565.569730,522.082216,525.469771,491.337221


In [70]:
rural.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 265 entries, 0 to 264
Data columns (total 23 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Country Name    265 non-null    object 
 1   Country Code    265 non-null    object 
 2   Indicator Name  265 non-null    object 
 3   Indicator Code  265 non-null    object 
 4   2000            264 non-null    float64
 5   2001            264 non-null    float64
 6   2002            264 non-null    float64
 7   2003            264 non-null    float64
 8   2004            264 non-null    float64
 9   2005            264 non-null    float64
 10  2006            264 non-null    float64
 11  2007            264 non-null    float64
 12  2008            264 non-null    float64
 13  2009            264 non-null    float64
 14  2010            264 non-null    float64
 15  2011            264 non-null    float64
 16  2012            264 non-null    float64
 17  2013            264 non-null    flo

In [71]:
rural.head(3)

,Country Name,Country Code,Indicator Name,Indicator Code,2000,2001,2002,2003,2004,2005,...,2009,2010,2011,2012,2013,2014,2015,2016,2017,2018
0,Aruba,ABW,Rural population (% of total population),SP.RUR.TOTL.ZS,34.645450,34.664886,34.717931,34.801708,34.911347,35.041977,...,35.677015,35.838703,36.000739,36.174791,36.361709,36.560747,36.771163,36.992212,37.223151,37.463236
1,Africa Eastern and Southern,AFE,Rural population (% of total population),SP.RUR.TOTL.ZS,71.531312,71.168494,70.778969,70.360964,69.912163,69.462724,...,67.759932,67.409639,67.061901,66.713294,66.332686,65.922423,65.509319,65.118789,64.717013,64.285282
2,Afghanistan,AFG,Rural population (% of total population),SP.RUR.TOTL.ZS,81.441800,81.258762,81.058752,80.839965,80.600641,80.294090,...,78.311452,77.738520,77.179431,76.656561,76.192286,75.808983,75.535088,75.341165,75.164718,75.000835


In [72]:
stabilite.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 265 entries, 0 to 264
Data columns (total 23 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Country Name    263 non-null    object 
 1   Country Code    265 non-null    object 
 2   Indicator Name  265 non-null    object 
 3   Indicator Code  265 non-null    object 
 4   2000            187 non-null    float64
 5   2001            0 non-null      float64
 6   2002            188 non-null    float64
 7   2003            198 non-null    float64
 8   2004            202 non-null    float64
 9   2005            202 non-null    float64
 10  2006            202 non-null    float64
 11  2007            202 non-null    float64
 12  2008            202 non-null    float64
 13  2009            203 non-null    float64
 14  2010            203 non-null    float64
 15  2011            204 non-null    float64
 16  2012            204 non-null    float64
 17  2013            204 non-null    flo

In [73]:
stabilite.head(3)

,Country Name,Country Code,Indicator Name,Indicator Code,2000,2001,2002,2003,2004,2005,...,2009,2010,2011,2012,2013,2014,2015,2016,2017,2018
0,Aruba,ABW,Political Stability - Governance score (0-100),GOV_WGI_PV_SC,NaN,NaN,NaN,NaN,76.642391,89.150306,...,88.557861,87.350033,87.265850,88.372221,88.34507,86.572494,86.551668,86.499810,86.506164,88.683848
1,NaN,AFE,Political Stability - Governance score (0-100),GOV_WGI_PV_SC,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Afghanistan,AFG,Political Stability - Governance score (0-100),GOV_WGI_PV_SC,17.203285,NaN,24.325111,25.113985,28.464202,29.442958,...,24.057531,22.661303,22.314866,23.121731,19.77865,26.287790,23.510621,23.472362,21.244270,21.782787


### <span style="color: SpringGreen;"> 4.2. Préparer les indicateurs pour 2017</span>

Pour chaque source, nous conservons le nom de l’entité, son code et la valeur de **2017**. Les colonnes sont renommées afin d’expliciter l’indicateur, l’année et l’unité.

In [74]:
pib_2017 = pib[["Country Name", "Country Code", "2017"]].copy()

pib_2017 = pib_2017.rename(columns={
    "Country Name": "Pays_BM",
    "Country Code": "Code ISO3",
    "2017": "PIB par habitant 2017 ($ US)"
})

In [75]:
rural_2017 = rural[["Country Name", "Country Code", "2017"]].copy()

rural_2017 = rural_2017.rename(columns={
    "Country Name": "Pays_BM",
    "Country Code": "Code ISO3",
    "2017": "Population rurale 2017 (%)"
})

Le **taux d’urbanisation** est calculé comme le complément à 100 de la part de population rurale. La colonne rurale est ensuite retirée pour ne pas conserver deux indicateurs qui apportent la même information.

In [76]:
rural_2017["Urbanisation 2017 (%)"] = (100 - rural_2017["Population rurale 2017 (%)"])

In [77]:
rural_2017.head(3)

,Pays_BM,Code ISO3,Population rurale 2017 (%),Urbanisation 2017 (%)
0,Aruba,ABW,37.223151,62.776849
1,Africa Eastern and Southern,AFE,64.717013,35.282987
2,Afghanistan,AFG,75.164718,24.835282


In [78]:
# Retirer la part rurale après le calcul de son complément, le taux d’urbanisation.
rural_2017 = rural_2017.drop(columns="Population rurale 2017 (%)")

In [79]:
rural_2017.head(3)

,Pays_BM,Code ISO3,Urbanisation 2017 (%)
0,Aruba,ABW,62.776849
1,Africa Eastern and Southern,AFE,35.282987
2,Afghanistan,AFG,24.835282


In [80]:
stabilite_2017 = stabilite[["Country Name", "Country Code", "2017"]].copy()

stabilite_2017 = stabilite_2017.rename(columns={
    "Country Name": "Pays_BM",
    "Country Code": "Code ISO3",
    "2017": "Stabilité politique 2017 (0-100)"
})

In [81]:
stabilite_2017.head(3)

,Pays_BM,Code ISO3,Stabilité politique 2017 (0-100)
0,Aruba,ABW,86.506164
1,NaN,AFE,NaN
2,Afghanistan,AFG,21.244270


### <span style="color: Lime;">4.3. Réunir les trois indicateurs de la Banque mondiale</span>

Les jointures sur « Code ISO3 » conservent les entrées du fichier de population rurale. Les validations d’unicité préviennent la multiplication des lignes.

In [83]:
# Réunir les trois indicateurs sur leur code commun.
bm_2017 = (rural_2017.merge(
        pib_2017[["Code ISO3", "PIB par habitant 2017 ($ US)"]],
        on="Code ISO3", how="left",validate="one_to_one" )
    .merge(stabilite_2017[["Code ISO3", "Stabilité politique 2017 (0-100)"]],
        on="Code ISO3", how="left", validate="one_to_one")
)

In [84]:
bm_2017.head(3)

,Pays_BM,Code ISO3,Urbanisation 2017 (%),PIB par habitant 2017 ($ US),Stabilité politique 2017 (0-100)
0,Aruba,ABW,62.776849,28440.041690,86.506164
1,Africa Eastern and Southern,AFE,35.282987,1529.923080,NaN
2,Afghanistan,AFG,24.835282,525.469771,21.244270


In [85]:
bm_2017.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 265 entries, 0 to 264
Data columns (total 5 columns):
 #   Column                            Non-Null Count  Dtype  
---  ------                            --------------  -----  
 0   Pays_BM                           265 non-null    object 
 1   Code ISO3                         265 non-null    object 
 2   Urbanisation 2017 (%)             264 non-null    float64
 3   PIB par habitant 2017 ($ US)      258 non-null    float64
 4   Stabilité politique 2017 (0-100)  204 non-null    float64
dtypes: float64(3), object(2)
memory usage: 10.5+ KB


In [86]:
bm_2017["Code ISO3"].duplicated().sum()

np.int64(0)

Le contrôle confirme l’absence de doublons dans la colonne « Code ISO3 ».

### <span style="color: LimeGreen;">4.4. Établir la correspondance entre les zones FAO et les codes de la Banque mondiale</span>

La table FAO utilise un **code de zone numérique**, tandis que les fichiers de la Banque mondiale utilisent des **codes à trois lettres**. Ces derniers comprennent aussi des codes d’agrégats, qui ne désignent pas des pays.

La table FAO comporte **172 pays et territoires**. La table fusionnée de la Banque mondiale contient **265 lignes** ; le fichier de PIB compte **263 noms distincts non manquants**. Ces effectifs ne représentent donc pas tous un nombre de pays.

La comparaison suivante repose sur les libellés exacts du fichier de PIB. Un libellé absent ne signifie pas nécessairement que le pays manque : son nom peut simplement être différent.

In [91]:
pays_fao = set(df_pays["Zone"])
pays_bm = set(pib_2017["Pays_BM"].dropna())

In [92]:
pays_fao_pas_bm = sorted(pays_fao - pays_bm)
pays_fao_pas_bm

['Bolivie (État plurinational de)',
 'Chine - RAS de Hong-Kong',
 'Chine - RAS de Macao',
 'Chine, Taiwan Province de',
 'Chine, continentale',
 'Congo',
 "Iran (République islamique d')",
 'Kirghizistan',
 "Royaume-Uni de Grande-Bretagne et d'Irlande du Nord",
 'République de Corée',
 'République de Moldova',
 'République populaire démocratique de Corée',
 'République-Unie de Tanzanie',
 'Slovaquie',
 'Tchéquie',
 'Venezuela (République bolivarienne du)',
 'Yémen',
 'Égypte',
 "États-Unis d'Amérique"]

In [93]:
pays_bm_pas_fao = sorted(pays_bm - pays_fao)
pays_bm_pas_fao

['Afrique du Nord et Moyen-Orient (BIRD et IDA)',
 'Afrique du Nord et Moyen-Orient (hors revenu élevé)',
 'Afrique subsaharienne',
 'Afrique subsaharienne (BIRD et IDA)',
 'Afrique subsaharienne (hors revenu élevé)',
 'Amérique du Nord',
 'Amérique latine et Caraïbes',
 'Amérique latine et Caraïbes (BIRD et IDA)',
 'Amérique latine et Caraïbes (hors revenu élevé)',
 'Andorre',
 'Aruba',
 'Asie de l’Est et Pacifique',
 'Asie de l’Est et Pacifique (BIRD et IDA)',
 'Asie de l’Est et Pacifique (hors revenu élevé)',
 'Asie du Sud',
 'Asie du Sud (BIRD et IDA)',
 'Autres petits états',
 'BIRD et IDA',
 'BIRD seulement',
 'Bahreïn',
 'Bermudes',
 'Bhoutan',
 'Bolivie',
 'Brunéi Darussalam',
 'Burundi',
 'Chine',
 'Chine, RAS de Hong Kong',
 'Cisjordanie et Gaza',
 'Comores',
 'Congo, République du',
 'Congo, République démocratique du',
 'Corée, République de',
 'Corée, République démocratique de',
 'Curacao',
 'Europe centrale et les pays baltes',
 'Europe et Asie centrale',
 'Europe et Asi

In [94]:
print("Pays FAO :", len(pays_fao))
print("Pays Banque mondiale :", len(pays_bm))
print("FAO absents de BM :", len(pays_fao_pas_bm))
print("BM absents de FAO :", len(pays_bm_pas_fao))

Pays FAO : 172
Pays Banque mondiale : 263
FAO absents de BM : 19
BM absents de FAO : 110


La liste de référence reste celle des **172 zones FAO**. Les fichiers de la Banque mondiale contiennent aussi des territoires et des agrégats géographiques ou économiques.

**19 libellés FAO** ne trouvent pas de correspondance exacte dans le fichier de PIB. Nous recherchons leurs variantes avant de compléter les codes.

In [95]:
# 120
# Attribuer les codes aux zones dont le libellé correspond exactement au fichier de PIB.
correspondance = df_pays[["Code zone", "Zone"]].merge(
    pib_2017[["Pays_BM", "Code ISO3"]],
    left_on="Zone",
    right_on="Pays_BM",
    how="left"
)

In [96]:
correspondance.head()

,Code zone,Zone,Pays_BM,Code ISO3
0,1,Arménie,Arménie,ARM
1,2,Afghanistan,Afghanistan,AFG
2,3,Albanie,Albanie,ALB
3,4,Algérie,Algérie,DZA
4,7,Angola,Angola,AGO


In [97]:
correspondance.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 172 entries, 0 to 171
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   Code zone  172 non-null    int64 
 1   Zone       172 non-null    object
 2   Pays_BM    153 non-null    object
 3   Code ISO3  153 non-null    object
dtypes: int64(1), object(3)
memory usage: 5.5+ KB


La jointure par libellé exact laisse **19 zones sans code**. Elles sont affichées ci-dessous.

In [98]:
correspondance[correspondance["Code ISO3"].isna()]

,Code zone,Zone,Pays_BM,Code ISO3
12,19,Bolivie (État plurinational de),NaN,NaN
26,41,"Chine, continentale",NaN,NaN
28,46,Congo,NaN,NaN
39,59,Égypte,NaN,NaN
61,96,Chine - RAS de Hong-Kong,NaN,NaN
67,102,Iran (République islamique d'),NaN,NaN
77,113,Kirghizistan,NaN,NaN
80,116,République populaire démocratique de Corée,NaN,NaN
81,117,République de Corée,NaN,NaN
89,128,Chine - RAS de Macao,NaN,NaN


Une recherche par fragments de noms aide à repérer les appellations utilisées dans le fichier de PIB. Chaque résultat doit être vérifié avant attribution du code ; la recherche par fragment ne constitue pas une correspondance automatique fiable.

In [99]:
recherches = ["boliv", "chine", "congo", "égypte","hong", "iran", "kirghi", "corée", "macao",
    "moldova", "tchè","slova", "taiwan", "tanzan", "royaume", "unis", "venezuela","yémen"]

for recherche in recherches:
    print("\nRecherche :", recherche)

    display(
        pib_2017[
            pib_2017["Pays_BM"]
            .str.contains(recherche, case=False, na=False)
        ][["Pays_BM", "Code ISO3"]]
    )


Recherche : boliv


,Pays_BM,Code ISO3
28,Bolivie,BOL



Recherche : chine


,Pays_BM,Code ISO3
40,Chine,CHN
95,"Chine, RAS de Hong Kong",HKG
145,"Région administrative spéciale de Macao, Chine",MAC



Recherche : congo


...

,Pays_BM,Code ISO3
8,Émirats arabes unis,ARE
242,Tunisie,TUN
250,États-Unis,USA



Recherche : venezuela


,Pays_BM,Code ISO3
253,Venezuela,VEN



Recherche : yémen


,Pays_BM,Code ISO3
261,"Yémen, Rép. du",YEM


Les correspondances vérifiées permettent d’attribuer manuellement **18 codes**. La zone « Chine, Taiwan Province de » ne possède pas d’entrée correspondante dans les fichiers utilisés ici ; son code reste manquant.

In [100]:
# Compléter les codes de 18 des 19 zones sans correspondance exacte.
# Taïwan reste sans correspondance dans les fichiers utilisés.

correspondance_iso3 = {
    "Bolivie (État plurinational de)": "BOL",
    "Chine - RAS de Hong-Kong": "HKG",
    "Chine - RAS de Macao": "MAC",
    "Chine, continentale": "CHN",
    "Congo": "COG",
    "Iran (République islamique d')": "IRN",
    "Kirghizistan": "KGZ",
    "Royaume-Uni de Grande-Bretagne et d'Irlande du Nord": "GBR",
    "République de Corée": "KOR",
    "République de Moldova": "MDA",
    "République populaire démocratique de Corée": "PRK",
    "République-Unie de Tanzanie": "TZA",
    "Slovaquie": "SVK",
    "Tchéquie": "CZE",
    "Venezuela (République bolivarienne du)": "VEN",
    "Yémen": "YEM",
    "Égypte": "EGY",
    "États-Unis d'Amérique": "USA"
}

In [101]:
correspondance["Code ISO3"] = correspondance["Code ISO3"].fillna(
    correspondance["Zone"].map(correspondance_iso3)
)

In [102]:
correspondance.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 172 entries, 0 to 171
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   Code zone  172 non-null    int64 
 1   Zone       172 non-null    object
 2   Pays_BM    153 non-null    object
 3   Code ISO3  171 non-null    object
dtypes: int64(1), object(3)
memory usage: 5.5+ KB


In [103]:
correspondance[correspondance["Code ISO3"].isna()]

,Code zone,Zone,Pays_BM,Code ISO3
145,214,"Chine, Taiwan Province de",NaN,NaN


In [104]:
# Conserver uniquement les identifiants nécessaires aux jointures.
correspondance = correspondance[["Code zone", "Zone", "Code ISO3"]]

In [105]:
correspondance.head()

,Code zone,Zone,Code ISO3
0,1,Arménie,ARM
1,2,Afghanistan,AFG
2,3,Albanie,ALB
3,4,Algérie,DZA
4,7,Angola,AGO


In [106]:
print(correspondance["Code zone"].duplicated().sum())
print(correspondance["Code ISO3"].duplicated().sum())
print(correspondance["Code ISO3"].isna().sum())

0
0
1


### <span style="color: YellowGreen;"> 4.5. Ajouter les codes et les indicateurs à la table FAO</span>

Deux jointures successives ajoutent les codes, puis les trois indicateurs de la Banque mondiale. Les jointures à gauche conservent les **172 zones FAO**, y compris celles dont certaines données sont manquantes.

In [107]:
# Ajouter le code de la Banque mondiale à chaque zone FAO.
df_pays = df_pays.merge(
    correspondance[["Code zone", "Code ISO3"]],
    on="Code zone",
    how="left",
    validate="one_to_one"
)

In [108]:
df_pays.head(3)

,Code zone,Zone,Disponibilité alimentaire (kcal/personne/jour),Disponibilité alimentaire (kg/personne/an),Disponibilité intérieure (milliers de tonnes),Importations (milliers de tonnes),Matières grasses (g/personne/jour),Nourriture (milliers de tonnes),Production (milliers de tonnes),Protéines (g/personne/jour),Population 2000 (milliers),Population 2017 (milliers),Croissance population 2000-2017 (%),Part importations dans disponibilité intérieure (%),Importations par habitant (kg/personne/an),Code ISO3
0,1,Arménie,54.0,16.06,47.0,35.0,3.39,47.0,11.0,5.44,3069.591,2944.791,-4.065688,74.468085,11.885394,ARM
1,2,Afghanistan,5.0,1.53,57.0,29.0,0.33,55.0,28.0,0.54,20779.953,36296.113,74.668889,50.877193,0.798984,AFG
2,3,Albanie,85.0,16.36,47.0,38.0,6.45,47.0,13.0,6.26,3129.243,2884.169,-7.831734,80.851064,13.175372,ALB


In [109]:
# Ajouter les trois indicateurs de la Banque mondiale en une seule jointure.
df_pays = df_pays.merge(
    bm_2017[[ "Code ISO3", "PIB par habitant 2017 ($ US)",
            "Urbanisation 2017 (%)", "Stabilité politique 2017 (0-100)"]],
    on="Code ISO3",  how="left", validate="one_to_one"
)

In [110]:
df_pays.head(3)

,Code zone,Zone,Disponibilité alimentaire (kcal/personne/jour),Disponibilité alimentaire (kg/personne/an),Disponibilité intérieure (milliers de tonnes),Importations (milliers de tonnes),Matières grasses (g/personne/jour),Nourriture (milliers de tonnes),Production (milliers de tonnes),Protéines (g/personne/jour),Population 2000 (milliers),Population 2017 (milliers),Croissance population 2000-2017 (%),Part importations dans disponibilité intérieure (%),Importations par habitant (kg/personne/an),Code ISO3,PIB par habitant 2017 ($ US),Urbanisation 2017 (%),Stabilité politique 2017 (0-100)
0,1,Arménie,54.0,16.06,47.0,35.0,3.39,47.0,11.0,5.44,3069.591,2944.791,-4.065688,74.468085,11.885394,ARM,3869.053739,64.712495,60.742784
1,2,Afghanistan,5.0,1.53,57.0,29.0,0.33,55.0,28.0,0.54,20779.953,36296.113,74.668889,50.877193,0.798984,AFG,525.469771,24.835282,21.244270
2,3,Albanie,85.0,16.36,47.0,38.0,6.45,47.0,13.0,6.26,3129.243,2884.169,-7.831734,80.851064,13.175372,ALB,5006.360130,55.980276,68.692869


## <span style="color: DarkViolet;">5. Contrôler la table finale</span>

### <span style="color: MediumOrchid;">5.1. Vérifier les dimensions et la couverture des indicateurs</span>


La table finale conserve 172 lignes. Les contrôles suivants détaillent les données de la Banque mondiale manquantes, puis les absences sur l’ensemble des colonnes.

In [111]:
df_pays.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 172 entries, 0 to 171
Data columns (total 19 columns):
 #   Column                                               Non-Null Count  Dtype  
---  ------                                               --------------  -----  
 0   Code zone                                            172 non-null    int64  
 1   Zone                                                 172 non-null    object 
 2   Disponibilité alimentaire (kcal/personne/jour)       172 non-null    float64
 3   Disponibilité alimentaire (kg/personne/an)           172 non-null    float64
 4   Disponibilité intérieure (milliers de tonnes)        170 non-null    float64
 5   Importations (milliers de tonnes)                    170 non-null    float64
 6   Matières grasses (g/personne/jour)                   172 non-null    float64
 7   Nourriture (milliers de tonnes)                      170 non-null    float64
 8   Production (milliers de tonnes)                      168 non-null    f

In [112]:
df_pays["Code ISO3"].isna().sum()

np.int64(1)

In [113]:
df_pays[
    [
        "PIB par habitant 2017 ($ US)",
        "Urbanisation 2017 (%)",
        "Stabilité politique 2017 (0-100)"
    ]
].isna().sum()

PIB par habitant 2017 ($ US)        2
Urbanisation 2017 (%)               1
Stabilité politique 2017 (0-100)    3
dtype: int64

In [114]:
# Afficher les zones présentant au moins un indicateur de la Banque mondiale manquant.

colonnes_bm = [
    "PIB par habitant 2017 ($ US)",
    "Urbanisation 2017 (%)",
    "Stabilité politique 2017 (0-100)"
]

df_pays.loc[
    df_pays[colonnes_bm].isna().any(axis=1),
    ["Zone", "Code ISO3"] + colonnes_bm
]

,Zone,Code ISO3,PIB par habitant 2017 ($ US),Urbanisation 2017 (%),Stabilité politique 2017 (0-100)
45,Polynésie française,PYF,20826.93125,61.789694,NaN
80,République populaire démocratique de Corée,PRK,NaN,61.820608,59.840915
106,Nouvelle-Calédonie,NCL,32354.71028,66.968339,NaN
145,"Chine, Taiwan Province de",NaN,NaN,NaN,NaN


### <span style="color: BlueViolet;">5.2. Examiner toutes les valeurs manquantes</span>

Le premier résultat donne le nombre de valeurs manquantes **par colonne**. Le tableau suivant affiche les **11 zones** concernées.

In [115]:
# Compter les valeurs manquantes par colonne, par ordre décroissant.

df_pays.isna().sum().sort_values(ascending=False)

Production (milliers de tonnes)                        4
Population 2000 (milliers)                             3
Stabilité politique 2017 (0-100)                       3
Croissance population 2000-2017 (%)                    3
PIB par habitant 2017 ($ US)                           2
Disponibilité intérieure (milliers de tonnes)          2
Part importations dans disponibilité intérieure (%)    2
Nourriture (milliers de tonnes)                        2
Importations par habitant (kg/personne/an)             2
Importations (milliers de tonnes)                      2
Urbanisation 2017 (%)                                  1
Code ISO3                                              1
Disponibilité alimentaire (kcal/personne/jour)         0
Code zone                                              0
Zone                                                   0
Protéines (g/personne/jour)                            0
Matières grasses (g/personne/jour)                     0
Disponibilité alimentaire (kg/p

In [116]:
df_pays.loc[
    df_pays.isna().any(axis=1)
]

,Code zone,Zone,Disponibilité alimentaire (kcal/personne/jour),Disponibilité alimentaire (kg/personne/an),Disponibilité intérieure (milliers de tonnes),Importations (milliers de tonnes),Matières grasses (g/personne/jour),Nourriture (milliers de tonnes),Production (milliers de tonnes),Protéines (g/personne/jour),Population 2000 (milliers),Population 2017 (milliers),Croissance population 2000-2017 (%),Part importations dans disponibilité intérieure (%),Importations par habitant (kg/personne/an),Code ISO3,PIB par habitant 2017 ($ US),Urbanisation 2017 (%),Stabilité politique 2017 (0-100)
45,70,Polynésie française,167.0,47.40,15.0,15.0,10.86,13.0,1.0,16.20,240.686,276.102,14.714607,100.000000,54.327748,PYF,20826.931250,61.789694,NaN
46,72,Djibouti,9.0,2.68,3.0,3.0,0.59,3.0,NaN,0.92,717.584,944.099,31.566339,100.000000,3.177633,DJI,2618.955211,71.318262,52.993998
80,116,République populaire démocratique de Corée,8.0,1.62,43.0,0.0,0.63,41.0,43.0,0.49,22929.075,25429.825,10.906458,0.000000,0.000000,PRK,NaN,61.820608,59.840915
84,120,République démocratique populaire lao,41.0,10.91,NaN,NaN,2.81,NaN,NaN,3.59,5323.700,6953.035,30.605312,NaN,NaN,LAO,2432.430595,34.511535,70.637255
93,132,Maldives,47.0,13.50,12.0,12.0,2.94,7.0,NaN,4.70,279.398,496.402,77.668416,100.000000,24.173956,MDV,10509.628810,39.327877,65.156839
106,153,Nouvelle-Calédonie,136.0,38.71,11.0,9.0,8.42,11.0,1.0,14.05,217.269,277.150,27.560766,81.818182,32.473390,NCL,32354.710280,66.968339,NaN
145,214,"Chine, Taiwan Province de",126.0,33.17,785.0,161.0,8.70,785.0,652.0,11.01,21966.527,23674.546,7.775553,20.509554,6.800553,NaN,NaN,NaN,NaN
160,235,Ouzbékistan,6.0,1.96,NaN,NaN,0.40,NaN,NaN,0.63,24769.955,31959.785,29.026415,NaN,NaN,UZB,2271.473011,50.678738,62.240996
169,272,Serbie,35.0,10.16,90.0,12.0,2.24,90.0,85.0,3.50,NaN,8829.628,NaN,13.333333,1.359061,SRB,6548.036538,60.621208,67.890373
170,273,Monténégro,55.0,15.98,10.0,8.0,3.37,10.0,4.0,5.79,NaN,627.563,NaN,80.000000,12.747724,MNE,7674.285657,64.999244,66.759443


## <span style="color: Brown;">6. Exporter la table pour l’analyse exploratoire</span>

La table exportée contient **172 pays et territoires et 19 colonnes** : **3 identifiants** (« Code zone », « Zone » et « Code ISO3 ») et **16 indicateurs**.

**11 lignes** présentent au moins une valeur manquante. Les données de population de 2017 sont complètes ; les autres absences sont conservées sans imputation ni suppression de zone.

Le fichier `df_pays.csv` servira à l’analyse exploratoire dans un second notebook. La sélection des indicateurs et le traitement des valeurs manquantes seront décidés à cette étape. L’export ci-dessous remplace le fichier du même nom dans le répertoire de travail.

In [ ]:
#df_pays.to_csv("df_pays.csv",index=False)